# A

```python

import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_log_error, r2_score
import json

import numpy as np

df = pd.read_csv("/content/train_weights.csv")

X = df.drop("MSE", axis=1)
y = np.log1p(df.MSE)

# Определение категориальных признаков
categorical_features_indices = np.where(X.dtypes != np.float64)[0]



model = CatBoostRegressor(iterations=2000,
                          learning_rate=0.03,
                          depth=6,
                          loss_function='RMSE',
                          verbose=False, # Отключаем вывод в процессе обучения
                          cat_features=categorical_features_indices)

# Обучение модели
model.fit(X, y)

# Предсказание на тестовых данных
predictions = model.predict(X)

# Оценка производительности модели
rmsle = root_mean_squared_log_error(y, predictions)
print(f"root_mean_squared_log_error: {rmsle}")
print(f"r2 {r2_score(y, predictions)}")
print(100*max(min((0.3 - rmsle) / 0.1, 1), 0))

X_test = pd.read_csv("/content/test_weights.csv")

predictions = model.predict(X_test)
X_test["MSE"] = np.expm1(predictions)

X_test.to_json('answers', orient='records')

with open('answers') as f:
  obja = json.load(f)

print(json.dumps(obja, indent=4))

```

# C

```cpp
#include <bits/stdc++.h>
using namespace std;
using pii = pair<int,int>;
const int INF = 1e9;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int n, m;
    if (!(cin >> n >> m)) return 0;
    int sx, sy;
    cin >> sx >> sy;
    --sx; --sy;
    vector<string> grid(n);
    for (int i = 0; i < n; ++i) cin >> grid[i];
    string s;
    cin >> s;
    if (s.empty()) { cout << 0 << '\n'; return 0; }

    // remove consecutive duplicates
    string ss;
    ss.push_back(s[0]);
    for (int i = 1; i < (int)s.size(); ++i)
        if (s[i] != ss.back()) ss.push_back(s[i]);
    s.swap(ss);

    int NM = n * m;
    auto idx = [&](int r, int c){ return r * m + c; };

    // locations as flat indices
    vector<vector<int>> loc(26);
    for (int r = 0; r < n; ++r)
        for (int c = 0; c < m; ++c)
            loc[grid[r][c] - 'a'].push_back(idx(r,c));

    // dp as flat array
    vector<int> dp(NM, INF), next_dp(NM, INF), trans(NM, INF);

    int first = s[0] - 'a';
    for (int p : loc[first]) {
        int r = p / m, c = p % m;
        dp[p] = abs(r - sx) + abs(c - sy);
    }

    auto manhattan_transform = [&](vector<int> &g) {
        // horizontal forward
        for (int r = 0; r < n; ++r) {
            int base = r * m;
            for (int c = 1; c < m; ++c) {
                int cur = base + c;
                int left = cur - 1;
                int nv = g[left] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
            for (int c = m - 2; c >= 0; --c) {
                int cur = base + c;
                int right = cur + 1;
                int nv = g[right] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
        }
        // vertical forward/back
        for (int c = 0; c < m; ++c) {
            for (int r = 1; r < n; ++r) {
                int cur = r * m + c;
                int up = cur - m;
                int nv = g[up] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
            for (int r = n - 2; r >= 0; --r) {
                int cur = r * m + c;
                int down = cur + m;
                int nv = g[down] + 1;
                if (nv < g[cur]) g[cur] = nv;
            }
        }
    };

    for (int i = 0; i + 1 < (int)s.size(); ++i) {
        int a = s[i] - 'a';
        int b = s[i+1] - 'a';
        const auto &A = loc[a];
        const auto &B = loc[b];

        // reset next_dp
        fill(next_dp.begin(), next_dp.end(), INF);

        long long prod = 1LL * A.size() * B.size();
        if (prod <= NM) {
            // brute pairwise
            for (int pa : A) {
                int valA = dp[pa];
                if (valA >= INF) continue;
                int rA = pa / m, cA = pa % m;
                for (int pb : B) {
                    int rB = pb / m, cB = pb % m;
                    int d = valA + abs(rA - rB) + abs(cA - cB);
                    if (d < next_dp[pb]) next_dp[pb] = d;
                }
            }
        } else {
            // transform approach
            // fill trans with INF and copy dp at A positions
            fill(trans.begin(), trans.end(), INF);
            for (int pa : A) {
                trans[pa] = dp[pa];
            }
            manhattan_transform(trans);
            for (int pb : B) next_dp[pb] = trans[pb];
        }
        dp.swap(next_dp);
    }

    int last = s.back() - 'a';
    int ans = INF;
    for (int p : loc[last]) if (dp[p] < ans) ans = dp[p];
    if (ans >= INF) ans = 0;
    cout << ans << '\n';
    return 0;
}

```